# MLP From Scratch Using NumPy — MNIST Digit Classification

**Goal:** Implement a multilayer perceptron with NumPy only for its neural-network computations. Train it to recognize digits 0–9.

**Architecture:** 784 → 128 (ReLU) → 64 (ReLU) → 10 (Softmax).

Run the cells in order in Google Colab (Runtime → Run all). No GPU is required.

## 1. Imports and reproducibility
We use TensorFlow **only to download MNIST**, not to build or train the model. NumPy implements all neural-network calculations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

SEED = 42
rng = np.random.default_rng(SEED)
print('NumPy version:', np.__version__)

## 2. Load and preprocess MNIST
MNIST has 60,000 training images and 10,000 test images. We reserve 5,000 of the training images for validation. Pixel values are scaled to [0, 1].

In [ ]:
(x_train_all, y_train_all), (x_test, y_test) = mnist.load_data()

x_train_all = x_train_all.reshape(-1, 784).astype(np.float32) / 255.0
x_test = x_test.reshape(-1, 784).astype(np.float32) / 255.0

indices = rng.permutation(len(x_train_all))
val_idx, train_idx = indices[:5000], indices[5000:]
X_train, y_train = x_train_all[train_idx], y_train_all[train_idx]
X_val, y_val = x_train_all[val_idx], y_train_all[val_idx]
X_test, y_test = x_test, y_test
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for ax, idx in zip(axes.flat, rng.choice(len(X_train), 10, replace=False)):
    ax.imshow(X_train[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'Label: {y_train[idx]}')
    ax.axis('off')
plt.suptitle('Sample MNIST images')
plt.tight_layout()
plt.show()

## 3. Core mathematics
For each dense layer, $Z = AW + b$. Hidden activations use $\mathrm{ReLU}(z)=\max(0,z)$.

Output probabilities use numerically stable softmax:
$$p_{ij}=\frac{e^{z_{ij}-m_i}}{\sum_k e^{z_{ik}-m_i}},\quad m_i=\max_k z_{ik}.$$

Mean categorical cross-entropy is
$$L=-\frac1N\sum_{i=1}^{N}\log(p_{i,y_i}).$$

For softmax plus cross-entropy, the output gradient is $(P-Y)/N$. Backpropagation applies the chain rule through each layer. We use **He initialization** for ReLU layers.

## 4. Implement the MLP entirely with NumPy

In [ ]:
class NumPyMLP:
    def __init__(self, layer_sizes=(784, 128, 64, 10), seed=42):
        generator = np.random.default_rng(seed)
        self.weights = []
        self.biases = []
        for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.weights.append((generator.standard_normal((fan_in, fan_out)) * np.sqrt(2 / fan_in)).astype(np.float32))
            self.biases.append(np.zeros((1, fan_out), dtype=np.float32))

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def softmax(z):
        shifted = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(shifted)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        activations = [X]
        pre_activations = []
        a = X
        for W, bias in zip(self.weights[:-1], self.biases[:-1]):
            z = a @ W + bias
            pre_activations.append(z)
            a = self.relu(z)
            activations.append(a)
        z = a @ self.weights[-1] + self.biases[-1]
        pre_activations.append(z)
        probabilities = self.softmax(z)
        activations.append(probabilities)
        return probabilities, activations, pre_activations

    @staticmethod
    def loss(probabilities, y):
        return -np.log(np.clip(probabilities[np.arange(len(y)), y], 1e-12, 1.0)).mean()

    def train_batch(self, X, y, learning_rate):
        probabilities, activations, pre_activations = self.forward(X)
        batch_loss = self.loss(probabilities, y)
        delta = probabilities.copy()
        delta[np.arange(len(y)), y] -= 1.0
        delta /= len(y)
        grad_W = [None] * len(self.weights)
        grad_b = [None] * len(self.biases)
        for layer in range(len(self.weights) - 1, -1, -1):
            grad_W[layer] = activations[layer].T @ delta
            grad_b[layer] = np.sum(delta, axis=0, keepdims=True)
            if layer > 0:
                delta = (delta @ self.weights[layer].T) * (pre_activations[layer - 1] > 0)
        for layer in range(len(self.weights)):
            self.weights[layer] -= learning_rate * grad_W[layer]
            self.biases[layer] -= learning_rate * grad_b[layer]
        return batch_loss

    def predict_proba(self, X):
        return self.forward(X)[0]

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def evaluate(self, X, y, batch_size=1024):
        total_loss, correct = 0.0, 0
        for start in range(0, len(X), batch_size):
            xb = X[start:start + batch_size]
            yb = y[start:start + batch_size]
            probabilities = self.predict_proba(xb)
            total_loss += self.loss(probabilities, yb) * len(yb)
            correct += np.sum(np.argmax(probabilities, axis=1) == yb)
        return total_loss / len(X), correct / len(X)

model = NumPyMLP()
print('Parameter count:', sum(w.size + b.size for w, b in zip(model.weights, model.biases)))

## 5. Train with mini-batch gradient descent
The validation set is used to monitor training. The test set remains untouched until final evaluation.

In [ ]:
EPOCHS = 15
BATCH_SIZE = 128
LEARNING_RATE = 0.1
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    permutation = rng.permutation(len(X_train))
    for start in range(0, len(X_train), BATCH_SIZE):
        batch_indices = permutation[start:start + BATCH_SIZE]
        model.train_batch(X_train[batch_indices], y_train[batch_indices], LEARNING_RATE)
    train_loss, train_acc = model.evaluate(X_train, y_train)
    val_loss, val_acc = model.evaluate(X_val, y_val)
    for key, value in zip(history, [train_loss, train_acc, val_loss, val_acc]):
        history[key].append(value)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train loss {train_loss:.4f} | train acc {train_acc:.2%} | val loss {val_loss:.4f} | val acc {val_acc:.2%}')

## 6. Plot learning curves

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Validation')
axes[0].set(title='Cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Validation')
axes[1].set(title='Classification accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Evaluate on the held-out test set

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.2%}')

## 8. Confusion matrix (NumPy only)
Rows represent true digits; columns represent predicted digits.

In [ ]:
predictions = model.predict(X_test)
confusion = np.zeros((10, 10), dtype=int)
np.add.at(confusion, (y_test, predictions), 1)
plt.figure(figsize=(8, 7))
plt.imshow(confusion, cmap='Blues')
plt.colorbar(label='Count')
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.title('MNIST Test Confusion Matrix')
for i in range(10):
    for j in range(10):
        plt.text(j, i, confusion[i, j], ha='center', va='center', fontsize=8,
                 color='white' if confusion[i, j] > confusion.max() / 2 else 'black')
plt.tight_layout(); plt.show()

## 9. Show example predictions

In [ ]:
sample_indices = rng.choice(len(X_test), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'True: {y_test[idx]} | Pred: {predictions[idx]}')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 10. Save and reload learned parameters
Save weights as a `.npz` file; this contains NumPy arrays and does not require TensorFlow.

In [ ]:
save_data = {}
for i, (W, bias) in enumerate(zip(model.weights, model.biases)):
    save_data[f'W{i}'] = W
    save_data[f'b{i}'] = bias
np.savez('numpy_mlp_mnist_weights.npz', **save_data)

reloaded = NumPyMLP()
with np.load('numpy_mlp_mnist_weights.npz') as data:
    for i in range(len(reloaded.weights)):
        reloaded.weights[i] = data[f'W{i}']
        reloaded.biases[i] = data[f'b{i}']
assert np.array_equal(reloaded.predict(X_test[:100]), model.predict(X_test[:100]))
print('Weights saved and reloaded successfully!')

## 11. Conclusion and ideas for experimentation

You built an MLP using **only NumPy** for weight initialization, forward propagation, softmax, cross-entropy, backpropagation, parameter updates, and inference.

**Try:** change the number of hidden neurons, learning rate, batch size, or epochs; compare validation curves; or add L2 regularization.

**Important:** The reported test accuracy is calculated only when you execute the notebook; results vary with settings and runtime.